# 2. Batch Correction Metrics, Downstream Modeling & Stress Test

Оценка структурных метрик батч-коррекции (AvgBATCH, AvgBio), обучение downstream ML-моделей на корректированных эмбеддингах, стресс-тест генерализации (Sanity → Weak OOD → True OOD).

**Пайплайн:** `0.eda_data_prep` → `1.batch_correction` → **`2.metrics_modeling_stress_test`**

**Вход:** `data/interim/corrected.adata.zarr` (из notebook 1)
**Выход:** `reports/` (сводные таблицы, фигуры), `data/processed/` (финальные результаты)

## 0. Imports & Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    SEED,
    DATA_INTERIM_DIR,
    DATA_PROCESSED_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    BATCH_COL,
    DIAGNOSIS_COL,
    SPLIT_PREFIX,
    TARGET_PREFIX,
    HARMONY_SUFFIX,
    DANN_SUFFIX,
    STRESS_LEVELS,
    CV_N_SPLITS,
    LAMA_TIMEOUT,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr

# Batch metrics
from batchcor_rna_emb.metrics.batch_metrics import (
    compute_kbet,
    compute_ilisi,
    compute_graph_connectivity,
    compute_asw_batch,
)

# Bio metrics
from batchcor_rna_emb.metrics.bio_metrics import (
    compute_clisi,
    compute_silhouette_bio,
    compute_nmi,
    compute_ari,
    run_leiden_clustering,
)

# Aggregation
from batchcor_rna_emb.metrics.aggregation import geometric_mean, build_comparison_table

# Modeling
from batchcor_rna_emb.modeling.train import train_tabpfn, train_lama, predict_proba
from batchcor_rna_emb.modeling.evaluation import (
    youden_threshold,
    evaluate_binary_classifier,
    compute_c_index,
)
from batchcor_rna_emb.modeling.feature_extraction import detect_pca_knee, fit_pca_pipeline, transform_pca_pipeline

# Stress test
from batchcor_rna_emb.stress_test.splits import prepare_stress_test_splits, log_split_summary

# Visualization
from batchcor_rna_emb.visualization.plots import (
    plot_umap_grid,
    plot_roc_curves,
    plot_metrics_heatmap,
    plot_scatter_avg_batch_bio,
    plot_generalization_decay,
)

# Setup
set_logger(level="INFO")
np.random.seed(SEED)

mpl.rcParams["figure.dpi"] = 150
sns.set_style("ticks")

In [ ]:
# Директории для выходных данных
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

METRICS_FIG_DIR = FIGURES_DIR / "metrics"
METRICS_FIG_DIR.mkdir(parents=True, exist_ok=True)

MODELING_FIG_DIR = FIGURES_DIR / "modeling"
MODELING_FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка corrected AnnData

In [ ]:
# Загрузка AnnData с корректированными эмбеддингами (из notebook 1)
adata = load_cohort(DATA_INTERIM_DIR / "corrected.adata.zarr")

# Извлечение метаданных из .uns
filtered_target_name = adata.uns.get("filtered_target_name", "Response_wo_SD")
target_col = f"{TARGET_PREFIX}{filtered_target_name}"
split_col = f"{SPLIT_PREFIX}{filtered_target_name}"

logger.info("Target: '{}', Split: '{}'", target_col, split_col)
logger.info("Shape: {}", adata.shape)
logger.info(".obsm ключи: {}", list(adata.obsm.keys()))

In [ ]:
# --- НАСТРОЙКА КЛЮЧЕЙ ЭМБЕДДИНГОВ (адаптировать под данные) ---
# Raw FM-эмбеддинг (до коррекции)
EMB_KEY = [k for k in adata.obsm.keys() if k.startswith("FM_") and HARMONY_SUFFIX not in k and DANN_SUFFIX not in k][0]

# Ключи корректированных эмбеддингов
HARMONY_KEY = f"{EMB_KEY}{HARMONY_SUFFIX}"
DANN_KEY = f"{EMB_KEY}{DANN_SUFFIX}"

# Словарь: метод коррекции → ключ .obsm
embedding_methods: dict[str, str] = {"Raw": EMB_KEY}
if HARMONY_KEY in adata.obsm:
    embedding_methods["Harmony"] = HARMONY_KEY
if DANN_KEY in adata.obsm:
    embedding_methods["DANN"] = DANN_KEY

logger.info("Методы коррекции и ключи: {}", embedding_methods)

## 2. Структурные метрики батч-коррекции

Вычисление метрик качества смешивания батчей (**AvgBATCH**) и сохранения биологического сигнала (**AvgBio**) для каждого метода коррекции.

**Batch metrics:** kBET, iLISI, Graph Connectivity, ASW Batch
**Bio metrics:** cLISI, Silhouette Bio, NMI, ARI

AvgBATCH и AvgBio — **геометрическое среднее** компонентных метрик (штрафует за нули).

In [ ]:
N_NEIGHBORS = 30

structural_results: dict[str, dict[str, float]] = {}

for method_name, emb_key in embedding_methods.items():
    logger.info("=== Вычисление метрик для метода: {} (key='{}') ===", method_name, emb_key)

    metrics: dict[str, float] = {}

    # --- Batch mixing metrics ---
    metrics["kBET"] = compute_kbet(adata, emb_key=emb_key, batch_col=BATCH_COL, n_neighbors=N_NEIGHBORS)
    metrics["iLISI"] = compute_ilisi(adata, emb_key=emb_key, batch_col=BATCH_COL, n_neighbors=N_NEIGHBORS)
    metrics["graph_connectivity"] = compute_graph_connectivity(adata, batch_col=BATCH_COL, emb_key=emb_key, n_neighbors=N_NEIGHBORS)
    metrics["ASW_batch"] = compute_asw_batch(adata, emb_key=emb_key, batch_col=BATCH_COL)

    # --- Bio preservation metrics ---
    # Leiden кластеризация — требуется для NMI и ARI
    cluster_key = f"leiden_{method_name}"
    run_leiden_clustering(adata, emb_key=emb_key, resolution=1.0, n_neighbors=N_NEIGHBORS, cluster_key=cluster_key)

    metrics["cLISI"] = compute_clisi(adata, emb_key=emb_key, label_col=DIAGNOSIS_COL, n_neighbors=N_NEIGHBORS)
    metrics["silhouette_bio"] = compute_silhouette_bio(adata, emb_key=emb_key, label_col=DIAGNOSIS_COL)
    metrics["NMI"] = compute_nmi(adata, label_col=DIAGNOSIS_COL, cluster_key=cluster_key)
    metrics["ARI"] = compute_ari(adata, label_col=DIAGNOSIS_COL, cluster_key=cluster_key)

    structural_results[method_name] = metrics
    logger.info("Метрики {}: {}", method_name, {k: f"{v:.4f}" for k, v in metrics.items()})

### 2.2 Сводная таблица структурных метрик

In [ ]:
# Сводная таблица с AvgBATCH и AvgBio
comparison_df = build_comparison_table(structural_results)
display(comparison_df.round(4))

# Сохранение в CSV
comparison_df.to_csv(REPORTS_DIR / "structural_metrics.csv")
logger.info("Сводная таблица структурных метрик сохранена в {}", REPORTS_DIR / "structural_metrics.csv")

### 2.3 UMAP-сравнение (до/после коррекции)

In [ ]:
# UMAP для каждого метода коррекции
for method_name, emb_key in embedding_methods.items():
    logger.info("Вычисление UMAP для метода: {} (key='{}')", method_name, emb_key)
    sc.pp.neighbors(adata, use_rep=emb_key, n_neighbors=N_NEIGHBORS, key_added=f"neighbors_{method_name}")
    sc.tl.umap(adata, neighbors_key=f"neighbors_{method_name}", random_state=SEED)
    adata.obsm[f"X_umap_{method_name}"] = adata.obsm["X_umap"].copy()

logger.info("UMAP вычислен для всех методов")

In [ ]:
# Подготовка словаря AnnData для plot_umap_grid:
# каждый метод → копия adata с соответствующим X_umap
umap_adatas: dict[str, ad.AnnData] = {}
for method_name in embedding_methods:
    adata_view = adata.copy()
    adata_view.obsm["X_umap"] = adata.obsm[f"X_umap_{method_name}"]
    umap_adatas[method_name] = adata_view

fig = plot_umap_grid(
    umap_adatas,
    color_cols=[BATCH_COL, DIAGNOSIS_COL],
    basis_key="X_umap",
    figsize=(6, 5),
    point_size=3,
    save_path=str(METRICS_FIG_DIR / "umap_correction_comparison.png"),
)
plt.show()
logger.info("UMAP-сравнение сохранено в {}", METRICS_FIG_DIR / "umap_correction_comparison.png")

## 3. Downstream Modeling

Обучение ML-моделей (TabPFN, LightAutoML) на корректированных эмбеддингах.
Оценка: бинарная классификация (F1-weighted, ROC-AUC) и survival analysis (C-index).

### 3.1 Подготовка фичей (PCA reduction)

In [ ]:
# --- Маски train/test ---
train_mask = adata.obs[split_col] == "train"
test_mask = adata.obs[split_col] == "test"

y_train = adata.obs.loc[train_mask, target_col].values.astype(int)
y_test = adata.obs.loc[test_mask, target_col].values.astype(int)

logger.info("Train: {} (pos={:.1%}), Test: {} (pos={:.1%})",
            train_mask.sum(), y_train.mean(), test_mask.sum(), y_test.mean())

# --- PCA-сжатие эмбеддингов для каждого метода ---
pca_features: dict[str, dict[str, np.ndarray]] = {}  # method → {"X_train": ..., "X_test": ...}

for method_name, emb_key in embedding_methods.items():
    X_full = np.asarray(adata.obsm[emb_key])
    X_tr = X_full[train_mask]
    X_te = X_full[test_mask]

    # Определение оптимального числа PCA-компонент по knee
    n_components = detect_pca_knee(X_tr, n_components=min(100, X_tr.shape[1]), seed=SEED, plot=False)
    logger.info("{}: PCA knee = {} компонент (из {})", method_name, n_components, X_tr.shape[1])

    scaler, pca = fit_pca_pipeline(X_tr, n_components=n_components, seed=SEED)
    X_train_pca = transform_pca_pipeline(X_tr, scaler, pca)
    X_test_pca = transform_pca_pipeline(X_te, scaler, pca)

    pca_features[method_name] = {"X_train": X_train_pca, "X_test": X_test_pca}
    logger.info("{}: PCA shapes — train {}, test {}", method_name, X_train_pca.shape, X_test_pca.shape)

### 3.2 Обучение моделей (TabPFN, LAMA)

In [ ]:
MODEL_FACTORIES = {
    "TabPFN": lambda X, y: train_tabpfn(X, y, seed=SEED),
    "LAMA": lambda X, y: train_lama(X, y, seed=SEED, timeout=LAMA_TIMEOUT),
}

# Результаты: (method, model_name) → {"model": ..., "y_prob": ..., "metrics": ...}
modeling_results: dict[tuple[str, str], dict] = {}

for method_name, feats in pca_features.items():
    X_tr, X_te = feats["X_train"], feats["X_test"]

    for model_name, factory in MODEL_FACTORIES.items():
        logger.info("--- Обучение {} на {} эмбеддингах ---", model_name, method_name)

        model = factory(X_tr, y_train)
        y_prob = predict_proba(model, X_te)

        # Youden threshold + evaluation
        thr, sens, spec = youden_threshold(y_test, y_prob)
        eval_metrics = evaluate_binary_classifier(y_test, y_prob, threshold=thr)

        modeling_results[(method_name, model_name)] = {
            "model": model,
            "y_prob": y_prob,
            "threshold": thr,
            "metrics": eval_metrics,
        }

        logger.info("{} + {}: ROC-AUC={:.4f}, F1-w={:.4f}, thr={:.3f}",
                    method_name, model_name,
                    eval_metrics.get("roc_auc", float("nan")),
                    eval_metrics.get("f1_weighted", float("nan")),
                    thr)

### 3.3 Evaluation: бинарная классификация

In [ ]:
# Сводная таблица метрик бинарной классификации
rows = []
for (method_name, model_name), res in modeling_results.items():
    m = res["metrics"]
    rows.append({
        "Method": method_name,
        "Model": model_name,
        "ROC-AUC": m.get("roc_auc"),
        "F1-weighted": m.get("f1_weighted"),
        "F1": m.get("f1"),
        "Precision": m.get("precision"),
        "Recall": m.get("recall"),
        "Threshold": res["threshold"],
    })

classification_df = pd.DataFrame(rows)
display(classification_df.round(4))
classification_df.to_csv(REPORTS_DIR / "classification_metrics.csv", index=False)
logger.info("Метрики классификации сохранены в {}", REPORTS_DIR / "classification_metrics.csv")

In [ ]:
# ROC-кривые: отдельный plot для каждого метода коррекции
for method_name in embedding_methods:
    roc_data: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    for model_name in MODEL_FACTORIES:
        key = (method_name, model_name)
        if key in modeling_results:
            roc_data[model_name] = (y_test, modeling_results[key]["y_prob"])

    fig = plot_roc_curves(
        roc_data,
        title=f"ROC — {method_name}",
        save_path=str(MODELING_FIG_DIR / f"roc_{method_name.lower()}.png"),
    )
    plt.show()

logger.info("ROC-кривые сохранены в {}", MODELING_FIG_DIR)

### 3.4 Evaluation: survival analysis (C-index)

`compute_c_index` принимает DataFrame с колонками duration, event и ковариатами (предсказанные вероятности).
Acceptance criterion: **C-index ≥ 0.75**.

In [ ]:
# --- Survival analysis: C-index ---
# Извлечение survival metadata из .uns (сохранено в notebook 0)
survival_duration_col = adata.uns.get("survival_duration_col", None)
survival_event_col = adata.uns.get("survival_event_col", None)

cindex_rows = []

if survival_duration_col and survival_event_col and \
   survival_duration_col in adata.obs.columns and survival_event_col in adata.obs.columns:

    logger.info("Survival data: duration='{}', event='{}'", survival_duration_col, survival_event_col)

    for (method_name, model_name), res in modeling_results.items():
        # Формируем design DataFrame для тестовой выборки
        design_df = adata.obs.loc[test_mask, [survival_duration_col, survival_event_col]].copy()
        design_df["pred_prob"] = res["y_prob"]
        design_df = design_df.dropna(subset=[survival_duration_col, survival_event_col])

        c_idx = compute_c_index(
            design_df,
            duration_col=survival_duration_col,
            event_col=survival_event_col,
            covariate_cols=["pred_prob"],
        )

        cindex_rows.append({
            "Method": method_name,
            "Model": model_name,
            "C-index": c_idx,
            "Criterion (≥0.75)": "PASS" if c_idx >= 0.75 else "FAIL",
        })

        logger.info("{} + {}: C-index = {:.4f} [{}]",
                    method_name, model_name, c_idx,
                    "PASS" if c_idx >= 0.75 else "FAIL")

    cindex_df = pd.DataFrame(cindex_rows)
    display(cindex_df.round(4))
    cindex_df.to_csv(REPORTS_DIR / "cindex_metrics.csv", index=False)
    logger.info("C-index сохранён в {}", REPORTS_DIR / "cindex_metrics.csv")
else:
    logger.warning("Survival данные не найдены в .uns или .obs — C-index пропущен")
    cindex_df = pd.DataFrame()

## 4. Stress Test генерализации

Три уровня стресс-теста:
1. **Sanity** — train/test из одной когорты (контроль переобучения)
2. **Weak OOD** — train на N-1 когортах, test на оставшейся (cross-batch)
3. **True OOD** — train на всех когортах, test на holdout-когорте (максимальный OOD)

**Важно:** `prepare_stress_test_splits()` извлекает `.X` (генную экспрессию). Для эмбеддингов используем маски train/test из splits и извлекаем фичи из `.obsm` вручную.

### 4.1 Определение holdout-когорт и создание сплитов

In [ ]:
# Определение holdout-когорт для True OOD
# Используем наименее представленную когорту как holdout
cohort_counts = adata.obs[BATCH_COL].value_counts()
logger.info("Когорты и размеры:\n{}", cohort_counts.to_string())

# Holdout — когорта с наименьшим числом образцов (или указать вручную)
holdout_cohorts = [cohort_counts.idxmin()]
logger.info("Holdout когорты для True OOD: {}", holdout_cohorts)

# Создание сплитов (на .X — для справки и логирования)
stress_splits = prepare_stress_test_splits(
    adata,
    target=filtered_target_name,
    holdout_cohorts=holdout_cohorts,
)

split_summary = log_split_summary(stress_splits)
display(split_summary)

### 4.2 Stress test loop: level × correction method × model

Для каждого уровня воспроизводим маски train/test, извлекаем эмбеддинги из `.obsm`, применяем PCA, обучаем модели.

In [ ]:
# --- Воспроизведение масок для каждого stress level ---
obs = adata.obs
target_col_full = f"{TARGET_PREFIX}{filtered_target_name}"
split_col_full = f"{SPLIT_PREFIX}{filtered_target_name}"

def _build_stress_masks(obs: pd.DataFrame, holdout_cohorts: list[str]) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Воссоздание bool-масок train/test для каждого stress level."""
    masks: dict[str, tuple[np.ndarray, np.ndarray]] = {}

    # Sanity: крупнейшая не-holdout когорта
    non_holdout = obs[~obs[BATCH_COL].isin(holdout_cohorts)]
    sanity_cohort = non_holdout[BATCH_COL].value_counts().idxmax()
    sanity_mask = obs[BATCH_COL] == sanity_cohort
    train_s = (sanity_mask & (obs[split_col_full] == "train")).values
    test_s = (sanity_mask & (obs[split_col_full] == "test")).values
    if train_s.sum() > 0 and test_s.sum() > 0:
        masks["sanity"] = (train_s, test_s)

    # Weak OOD: все не-holdout когорты
    weak_mask = ~obs[BATCH_COL].isin(holdout_cohorts)
    train_w = (weak_mask & (obs[split_col_full] == "train")).values
    test_w = (weak_mask & (obs[split_col_full] == "test")).values
    if train_w.sum() > 0 and test_w.sum() > 0:
        masks["weak_ood"] = (train_w, test_w)

    # True OOD: train = не-holdout train, test = holdout с валидным таргетом
    train_t = (~obs[BATCH_COL].isin(holdout_cohorts) & (obs[split_col_full] == "train")).values
    test_t = (obs[BATCH_COL].isin(holdout_cohorts) & obs[target_col_full].notna()).values
    if train_t.sum() > 0 and test_t.sum() > 0:
        masks["true_ood"] = (train_t, test_t)

    return masks

stress_masks = _build_stress_masks(obs, holdout_cohorts)
logger.info("Stress masks готовы: {}", {k: (v[0].sum(), v[1].sum()) for k, v in stress_masks.items()})

In [ ]:
# --- Основной цикл стресс-теста ---
stress_results: list[dict] = []

for level_name, (train_idx, test_idx) in stress_masks.items():
    y_tr_stress = obs.loc[train_idx, target_col_full].values.astype(int)
    y_te_stress = obs.loc[test_idx, target_col_full].values.astype(int)

    for method_name, emb_key in embedding_methods.items():
        X_full = np.asarray(adata.obsm[emb_key])
        X_tr = X_full[train_idx]
        X_te = X_full[test_idx]

        # PCA
        n_comp = detect_pca_knee(X_tr, n_components=min(100, X_tr.shape[1]), seed=SEED, plot=False)
        scaler, pca = fit_pca_pipeline(X_tr, n_components=n_comp, seed=SEED)
        X_tr_pca = transform_pca_pipeline(X_tr, scaler, pca)
        X_te_pca = transform_pca_pipeline(X_te, scaler, pca)

        for model_name, factory in MODEL_FACTORIES.items():
            logger.info("[{}] {} + {} — обучение...", level_name, method_name, model_name)

            model = factory(X_tr_pca, y_tr_stress)
            y_prob = predict_proba(model, X_te_pca)
            thr, _, _ = youden_threshold(y_te_stress, y_prob)
            metrics = evaluate_binary_classifier(y_te_stress, y_prob, threshold=thr)

            stress_results.append({
                "Level": level_name,
                "Method": method_name,
                "Model": model_name,
                "ROC-AUC": metrics.get("roc_auc"),
                "F1-weighted": metrics.get("f1_weighted"),
                "F1": metrics.get("f1"),
                "N_train": int(train_idx.sum()),
                "N_test": int(test_idx.sum()),
            })

            logger.info("[{}] {} + {}: ROC-AUC={:.4f}, F1-w={:.4f}",
                        level_name, method_name, model_name,
                        metrics.get("roc_auc", float("nan")),
                        metrics.get("f1_weighted", float("nan")))

### 4.3 Сводные таблицы стресс-теста

In [ ]:
stress_df = pd.DataFrame(stress_results)
display(stress_df.round(4))

stress_df.to_csv(REPORTS_DIR / "stress_test_metrics.csv", index=False)
logger.info("Стресс-тест метрики сохранены в {}", REPORTS_DIR / "stress_test_metrics.csv")

In [ ]:
# Generalization decay plot: для каждого метода коррекции
for method_name in embedding_methods:
    method_df = stress_df[stress_df["Method"] == method_name].copy()
    method_df = method_df.rename(columns={"Level": "level", "Model": "model", "F1-weighted": "f1_weighted"})

    if not method_df.empty:
        fig = plot_generalization_decay(
            method_df,
            metric_col="f1_weighted",
            level_col="level",
            group_col="model",
            save_path=str(MODELING_FIG_DIR / f"decay_{method_name.lower()}.png"),
        )
        fig.suptitle(f"Generalization Decay — {method_name}", fontsize=14, y=1.02)
        plt.show()

logger.info("Графики generalization decay сохранены в {}", MODELING_FIG_DIR)

## 5. Итоговые результаты

### 5.1 Сводная таблица: структурные метрики + downstream performance

In [ ]:
# Объединённая таблица: структурные метрики + лучшая модель для каждого метода на каждом уровне
summary_rows = []

for method_name in embedding_methods:
    row: dict[str, object] = {"Method": method_name}

    # Структурные метрики
    if method_name in structural_results:
        struct = structural_results[method_name]
        batch_keys = ["kBET", "iLISI", "graph_connectivity", "ASW_batch"]
        bio_keys = ["cLISI", "silhouette_bio", "NMI", "ARI"]
        row["AvgBATCH"] = geometric_mean([struct[k] for k in batch_keys])
        row["AvgBio"] = geometric_mean([struct[k] for k in bio_keys])

    # Downstream: лучший ROC-AUC на основном split (из classification_df)
    method_cls = classification_df[classification_df["Method"] == method_name]
    if not method_cls.empty:
        best_row = method_cls.loc[method_cls["ROC-AUC"].idxmax()]
        row["Best Model"] = best_row["Model"]
        row["ROC-AUC (main)"] = best_row["ROC-AUC"]
        row["F1-w (main)"] = best_row["F1-weighted"]

    # Stress test: F1-w по уровням (лучшая модель)
    for level in STRESS_LEVELS:
        level_data = stress_df[(stress_df["Method"] == method_name) & (stress_df["Level"] == level)]
        if not level_data.empty:
            row[f"F1-w ({level})"] = level_data["F1-weighted"].max()

    # C-index
    if not cindex_df.empty:
        ci_method = cindex_df[cindex_df["Method"] == method_name]
        if not ci_method.empty:
            row["C-index (best)"] = ci_method["C-index"].max()

    summary_rows.append(row)

final_summary = pd.DataFrame(summary_rows).set_index("Method")
display(final_summary.round(4))

final_summary.to_csv(REPORTS_DIR / "final_summary.csv")
logger.info("Итоговая сводная таблица сохранена в {}", REPORTS_DIR / "final_summary.csv")

### 5.2 Визуализации

In [ ]:
# Heatmap структурных метрик
fig = plot_metrics_heatmap(
    comparison_df,
    title="Structural Metrics: Batch Correction Comparison",
    save_path=str(METRICS_FIG_DIR / "structural_metrics_heatmap.png"),
)
plt.show()

# Scatter AvgBATCH vs AvgBio
fig = plot_scatter_avg_batch_bio(
    comparison_df,
    save_path=str(METRICS_FIG_DIR / "avgbatch_vs_avgbio.png"),
)
plt.show()

In [ ]:
# Сводный generalization decay: все методы, группировка по Method+Model
decay_df = stress_df.copy()
decay_df = decay_df.rename(columns={"Level": "level", "F1-weighted": "f1_weighted"})
decay_df["group"] = decay_df["Method"] + " + " + decay_df["Model"]

fig = plot_generalization_decay(
    decay_df,
    metric_col="f1_weighted",
    level_col="level",
    group_col="group",
    save_path=str(MODELING_FIG_DIR / "generalization_decay_all.png"),
)
plt.show()
logger.info("Все визуализации сохранены")

### 5.3 Аналитический вывод

**Ключевые вопросы:**
1. Улучшает ли батч-коррекция структурные метрики (AvgBATCH ↑) без потери биологического сигнала (AvgBio)?
2. Как методы коррекции влияют на downstream classification (ROC-AUC, F1-weighted)?
3. Какой метод наиболее устойчив к generalization decay (Sanity → Weak OOD → True OOD)?
4. Достигается ли клинически значимый порог C-index ≥ 0.75?

**Интерпретация:**
- Если AvgBATCH растёт, а AvgBio падает — коррекция стирает биологический сигнал.
- Если F1-w на True OOD значительно ниже Sanity — модель не генерализуется (high generalization decay).
- Оптимальный метод: максимальный AvgBATCH при минимальной потере AvgBio и стабильный downstream performance.

> Заполнить выводы после запуска пайплайна на реальных данных.

## 6. Сохранение финальных результатов

In [ ]:
# Сохранение AnnData с UMAP-координатами и кластеризацией
save_adata_zarr(adata, DATA_PROCESSED_DIR / "final.adata.zarr")
logger.info("Финальный AnnData сохранён в {}", DATA_PROCESSED_DIR / "final.adata.zarr")

# Перечень всех сохранённых отчётов
logger.info("=== Сохранённые файлы ===")
logger.info("Отчёты:")
for f in sorted(REPORTS_DIR.glob("*.csv")):
    logger.info("  {}", f.name)
logger.info("Фигуры (metrics):")
for f in sorted(METRICS_FIG_DIR.glob("*.png")):
    logger.info("  {}", f.name)
logger.info("Фигуры (modeling):")
for f in sorted(MODELING_FIG_DIR.glob("*.png")):
    logger.info("  {}", f.name)